In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X_train = np.load("../data/processed/X_train_binary.npy")
X_val = np.load("../data/processed/X_val_binary.npy")
y_train = pd.read_csv("../data/processed/y_train_binary.csv")['Label_binary']
y_val = pd.read_csv("../data/processed/y_val_binary.csv")['Label_binary']

print(X_train.shape, y_train.shape)

(1764506, 78) (1764506,)


In [2]:
clf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,      # use all CPU cores
    max_depth=20    # prevents runaway memory/time on 1.7M rows
)

clf.fit(X_train, y_train)
print("Training complete.")

Training complete.


In [3]:
y_val_binary_numeric = (y_val == 'ATTACK').astype(int)
y_pred_proba_attack = clf.predict_proba(X_val)[:, list(clf.classes_).index('ATTACK')]

print(f"ROC-AUC: {roc_auc_score(y_val_binary_numeric, y_pred_proba_attack):.4f}")

ROC-AUC: 1.0000


In [4]:
df_check = pd.read_parquet("../data/processed/cleaned_flows.parquet")
feature_names = [c for c in df_check.columns if c not in ['Label', 'Label_binary']]

importances = pd.Series(clf.feature_importances_, index=feature_names)
print(importances.sort_values(ascending=False).head(15))

Max Packet Length              0.082159
Packet Length Std              0.071082
Packet Length Variance         0.067345
Average Packet Size            0.058965
Avg Bwd Segment Size           0.052264
Bwd Packet Length Max          0.040250
Destination Port               0.037704
Bwd Packet Length Std          0.036932
Total Length of Bwd Packets    0.029588
Packet Length Mean             0.028655
Fwd Packet Length Max          0.027573
Init_Win_bytes_backward        0.025195
Fwd Packet Length Mean         0.024361
Subflow Fwd Bytes              0.023054
Init_Win_bytes_forward         0.022316
dtype: float64


In [5]:
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

      ATTACK       0.99      1.00      1.00     63851
      BENIGN       1.00      1.00      1.00    314258

    accuracy                           1.00    378109
   macro avg       1.00      1.00      1.00    378109
weighted avg       1.00      1.00      1.00    378109


Confusion matrix:
[[ 63760     91]
 [   394 313864]]


In [6]:
# Hash each row (excluding label columns) to detect near-identical flows
# that survived exact-dedup but are still suspiciously similar across splits
train_hashes = set(pd.util.hash_pandas_object(pd.DataFrame(X_train)).values)
val_hashes = set(pd.util.hash_pandas_object(pd.DataFrame(X_val)).values)

overlap = train_hashes & val_hashes
print(f"Exact numeric-row overlap between train and val: {len(overlap)} rows")
print(f"As % of val set: {len(overlap)/len(X_val)*100:.4f}%")

Exact numeric-row overlap between train and val: 0 rows
As % of val set: 0.0000%


In [7]:
X_test = np.load("../data/processed/X_test_binary.npy")
y_test = pd.read_csv("../data/processed/y_test_binary.csv")['Label_binary']

y_test_pred = clf.predict(X_test)
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

      ATTACK       0.99      1.00      1.00     63851
      BENIGN       1.00      1.00      1.00    314258

    accuracy                           1.00    378109
   macro avg       1.00      1.00      1.00    378109
weighted avg       1.00      1.00      1.00    378109



In [8]:
joblib.dump(clf, "../data/processed/rf_baseline_binary.joblib")
print("Model saved.")

Model saved.


In [9]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("network-anomaly-binary")

with mlflow.start_run(run_name="rf_baseline"):
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("val_roc_auc", roc_auc_score(y_val_binary_numeric, y_pred_proba_attack))
    mlflow.sklearn.log_model(clf, "model")
    print("Logged to MLflow.")

2026/09/13 22:54:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged to MLflow.
🏃 View run rf_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/df486ae7a9094d9e9a174a65bc7fdec9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [10]:
from xgboost import XGBClassifier

# convert labels to numeric for XGBoost
y_train_numeric = (y_train == 'ATTACK').astype(int)
y_val_numeric = (y_val == 'ATTACK').astype(int)

# scale_pos_weight = ratio of negative to positive class, XGBoost's equivalent of class_weight='balanced'
scale_pos_weight = (y_train_numeric == 0).sum() / (y_train_numeric == 1).sum()

xgb_clf = XGBClassifier(
    n_estimators=100,
    max_depth=10,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

xgb_clf.fit(X_train, y_train_numeric)
print("XGBoost training complete.")

XGBoost training complete.


In [11]:
y_val_pred_xgb = xgb_clf.predict(X_val)
y_val_proba_xgb = xgb_clf.predict_proba(X_val)[:, 1]

print(classification_report(y_val_numeric, y_val_pred_xgb, target_names=['BENIGN', 'ATTACK']))
print(f"ROC-AUC: {roc_auc_score(y_val_numeric, y_val_proba_xgb):.4f}")

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    314258
      ATTACK       1.00      1.00      1.00     63851

    accuracy                           1.00    378109
   macro avg       1.00      1.00      1.00    378109
weighted avg       1.00      1.00      1.00    378109

ROC-AUC: 1.0000


In [12]:
from lightgbm import LGBMClassifier

lgbm_clf = LGBMClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_clf.fit(X_train, y_train_numeric)
print("LightGBM training complete.")

LightGBM training complete.


In [13]:
y_val_pred_lgbm = lgbm_clf.predict(X_val)
y_val_proba_lgbm = lgbm_clf.predict_proba(X_val)[:, 1]

print(classification_report(y_val_numeric, y_val_pred_lgbm, target_names=['BENIGN', 'ATTACK']))
print(f"ROC-AUC: {roc_auc_score(y_val_numeric, y_val_proba_lgbm):.4f}")

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    314258
      ATTACK       0.99      1.00      1.00     63851

    accuracy                           1.00    378109
   macro avg       1.00      1.00      1.00    378109
weighted avg       1.00      1.00      1.00    378109

ROC-AUC: 1.0000


In [14]:
with mlflow.start_run(run_name="xgboost_baseline"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("val_roc_auc", roc_auc_score(y_val_numeric, y_val_proba_xgb))
    mlflow.xgboost.log_model(xgb_clf, "model")

with mlflow.start_run(run_name="lightgbm_baseline"):
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("val_roc_auc", roc_auc_score(y_val_numeric, y_val_proba_lgbm))
    mlflow.lightgbm.log_model(lgbm_clf, "model")

print("Both models logged to MLflow.")

2026/09/13 22:55:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run xgboost_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/7a1e03de1da64247947da9ce5560b2eb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/09/13 22:55:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run lightgbm_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/3a0c3874b0ea45d6ae8ba6d11bf6ee1d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Both models logged to MLflow.
